# 6.3 Apply the dedup rule and re-measure MAE

Same split and CatBoost settings as 5.3. Duplicates are removed after the split, so train / val / test keep the same groups. Needs `dup_rate_6_2.csv` from 6.2.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/processed/task6/dup_candidates_6_1.csv')
rate = pd.read_csv('../data/processed/task6/dup_rate_6_2.csv')

apply_rule = bool(rate.loc[0, 'apply_rule'])
print("Dataset shape:", df.shape)
print("Duplicate rate from 6.2:", rate.loc[0, 'dataset_rate'])
print("Apply dedup rule (rate >= 5%):", apply_rule)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

features = [
    'model_key', 'brand', 'family', 'storage_gb', 'ram_gb',
    'condition', 'has_5g', 'has_warranty', 'has_box',
    'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint',
    'is_tablet_hint', 'is_parts_hint', 'accessory_first',
    'bundle_hint', 'is_feature_phone_hint', 'loc_area',
    'loc_city', 'storage_known', 'ram_known', 'condition_known'
]
cat_features = ['model_key', 'brand', 'family', 'condition', 'loc_area', 'loc_city']

catboost_params = {
    'iterations': 1500,
    'learning_rate': 0.03,
    'depth': 6,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'verbose': 0,
    'allow_writing_files': False
}

In [ ]:
def split_data(data, seed):
    # 70% train, 15% validation, 15% test, grouped by title_group (as in 5.3)
    train_idx, temp_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed).split(data, groups=data['title_group']))
    temp = data.iloc[temp_idx]
    val_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed).split(temp, groups=temp['title_group']))
    return data.iloc[train_idx], temp.iloc[val_idx], temp.iloc[test_idx]


def drop_duplicates(train, val, test):
    # keep the first ad of each group inside every part
    train = train[train['dup_candidate_id'].isna() | ~train.sort_values('ad_id').duplicated('dup_candidate_id')]
    val = val[val['dup_candidate_id'].isna() | ~val.sort_values('ad_id').duplicated('dup_candidate_id')]
    test = test[test['dup_candidate_id'].isna() | ~test.sort_values('ad_id').duplicated('dup_candidate_id')]

    # same group in two parts -> remove it from the earlier part, so nothing leaks into test
    val = val[~val['dup_candidate_id'].isin(test['dup_candidate_id'].dropna())]
    train = train[~train['dup_candidate_id'].isin(pd.concat([val, test])['dup_candidate_id'].dropna())]
    return train, val, test


def prepare(data):
    X = data[features].copy()
    X[cat_features] = X[cat_features].fillna('Unknown').astype(str)
    return X


def train_model(train, val):
    # half of the validation groups are used for early stopping (as in 5.3)
    stop_idx, _ = next(GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42).split(val, groups=val['title_group']))
    early_stop = val.iloc[stop_idx]
    early_stop = early_stop[early_stop['listing_type'] == 'phone']

    model = CatBoostRegressor(loss_function='Quantile:alpha=0.5', **catboost_params)
    model.fit(prepare(train), train['price'], cat_features=cat_features,
              eval_set=(prepare(early_stop), early_stop['price']), early_stopping_rounds=100)
    return model

In [ ]:
results = []

if apply_rule:
    for seed in [42, 7, 2026]:
        train, val, test = split_data(df, seed)
        train_d, val_d, test_d = drop_duplicates(train, val, test)

        test_phone = test[test['listing_type'] == 'phone']
        test_phone_d = test_d[test_d['listing_type'] == 'phone']

        model_before = train_model(train, val)
        model_after = train_model(train_d, val_d)

        results.append({
            'seed': seed,
            'train': len(train),
            'train_dedup': len(train_d),
            'test_phone': len(test_phone),
            'test_phone_dedup': len(test_phone_d),
            'MAE_before': mean_absolute_error(test_phone['price'], model_before.predict(prepare(test_phone))),
            'MAE_before_on_dedup_test': mean_absolute_error(test_phone_d['price'], model_before.predict(prepare(test_phone_d))),
            'MAE_after': mean_absolute_error(test_phone_d['price'], model_after.predict(prepare(test_phone_d))),
        })

    results = pd.DataFrame(results)
    display(results.round(1))
    display(results.drop(columns='seed').mean().round(1).to_frame('mean').T)
else:
    print("Rate is below 5%: dedup rule is not applied, nothing to re-measure.")

`MAE_before` is the current model on the current test. `MAE_before_on_dedup_test` is the same model on the cleaned test (same rows as `MAE_after`), so `MAE_before_on_dedup_test` vs `MAE_after` is the fair comparison.

In [ ]:
if apply_rule:
    results.to_csv('../data/processed/task6/dup_mae_effect_6_3.csv', index=False)
    df_dedup = df[df['dup_candidate_id'].isna() | ~df.sort_values('ad_id').duplicated('dup_candidate_id')]
    df_dedup.to_csv('../data/processed/task6/train_dedup_6_3.csv', index=False)
    print("Deduplicated dataset:", df_dedup.shape)
else:
    print("Nothing to save: rule not applied.")